# Dofus THX-01 Fine-Tuning (Colab / Kaggle)

Fine-tunes **doofz/THX-01** (322M encoder-only decision model, ~10ms latency) on Dofus game data:
combat actions, item lookup, map coordinates, NPC dialogue.

THX-01 uses RLCD (Reinforcement Learning for Calibrated Decisions), not causal-LM fine-tuning.
Run this on Colab GPU or Kaggle GPU.


In [ ]:
# Install dependencies
!pip install -q thx01[train] fastapi uvicorn datasets huggingface_hub

import os
os.environ["HF_TOKEN"] = "<your-hf-token>"  # set in Colab secrets


In [ ]:
# Mount Google Drive (Colab) or skip on Kaggle
from google.colab import drive
drive.mount("/content/drive")

# Your local pipeline files uploaded to Drive, or clone from GitHub
%cd /content/drive/MyDrive/dofus-emu/dofus-network
!pwd
!ls -la pipeline/

In [ ]:
# Step 1: Filter raw Dofus extracts into THX-01 training JSONL
from pipeline.data_filter import filter_directory

written = filter_directory(raw_dir="data/raw", out="data/processed")
print(f"Processed {len(written)} files:")
for w in written:
    print(" -", w)

In [ ]:
# Step 2: Fine-tune doofz/THX-01 with RLCD on Dofus data
from pipeline.train_thx01 import train

train(
    data_files=["data/processed/*.jsonl"],
    base_model="doofz/THX-01",
    output_dir="models/finetuned/thx01-dofus",
    epochs=3,
    batch_size=16,
    learning_rate=1e-5,
    push_to_hub=None,  # set to "your-username/thx01-dofus" to upload
)

In [ ]:
# Step 3: Benchmark base vs fine-tuned
from pipeline.benchmark import benchmark_model, compare_reports, load_model
import json

suite = [  # same Dofus suite as pipeline/benchmark.py
    {
        "id": "dofus_combat_001",
        "state": "Combat turn 3. Iop vs Goblin. Iop has 120/150 PV, 6/6 PA, 3/3 PM. Goblin at distance 2. Available spells: Coup d'Epee (1 PA, 15-20 dmg), Pulsation (2 PA, 30 dmg, AoE).",
        "questions": {
            "combat_action": {
                "type": "choice",
                "question": "What combat action should the character take next?",
                "criteria": {
                    "attack": "Use an offensive spell or weapon",
                    "defend": "Use a defensive or healing action",
                    "move": "Change position on the map",
                    "item": "Use an item from inventory",
                    "pass": "Skip turn / wait",
                },
            },
        },
        "expected": {"combat_action": "attack"},
    },
]

base_agent = load_model("doofz/THX-01")
base_report = benchmark_model(base_agent, suite)

ft_agent = load_model("models/finetuned/thx01-dofus")
ft_report = benchmark_model(ft_agent, suite)

comp = compare_reports(base_report, ft_report)
print(json.dumps(comp, indent=2))


In [ ]:
# Step 4: Start local inference server (~10ms target)
!python pipeline/serve_thx01.py --model models/finetuned/thx01-dofus --port 8095 &

# Test it
import requests
r = requests.post("http://localhost:8095/v1/decide", json={
    "state": "Combat turn 3. Iop vs Goblin. Iop has 120/150 PV, 6/6 PA.",
    "questions": {
        "combat_action": {
            "type": "choice",
            "question": "What combat action should the character take next?",
            "criteria": {
                "attack": "Use an offensive spell or weapon",
                "defend": "Use a defensive or healing action",
                "move": "Change position on the map",
                "item": "Use an item from inventory",
                "pass": "Skip turn / wait",
            },
        },
    },
})
print(r.json())
